In [ ]:
#Install dependencies
%pip install anthropic python-dotenv

In [1]:
#Load donenv
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
#Create API client
from anthropic import Anthropic
client = Anthropic()
model = "claude-haiku-4-5"

In [3]:
def add_user_message(messages, text):
    user_message = {"role":"user", "content":text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role":"assistant", "content":text}
    messages.append(assistant_message)

#Make a request
def chat(messages, system = None, stop_sequences = []):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [4]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [ ]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [5]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:
{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [6]:
def grade_by_model(test_case, output):
    eval_prompt=f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
"""
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```code")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [7]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # ToDo Grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return{
        "output":output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [8]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and runs run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean(result["score"] for result in results)
    print(f"Average score: {average_score}")

    return results

In [9]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.333333333333333


In [10]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Python Function to Extract S3 Bucket Name from ARN\n\n```python\ndef extract_bucket_name_from_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the S3 bucket name from an S3 bucket ARN.\n    \n    Args:\n        arn (str): S3 bucket ARN in the format 'arn:aws:s3:::bucket-name'\n    \n    Returns:\n        str: The bucket name\n    \n    Raises:\n        ValueError: If the ARN format is invalid\n    \n    Example:\n        >>> extract_bucket_name_from_arn('arn:aws:s3:::my-bucket-name')\n        'my-bucket-name'\n    \"\"\"\n    # Validate ARN format\n    if not isinstance(arn, str) or not arn.startswith('arn:aws:s3:::'):\n        raise ValueError(f\"Invalid S3 bucket ARN format: {arn}\")\n    \n    # Extract bucket name (everything after 'arn:aws:s3:::')\n    bucket_name = arn[len('arn:aws:s3:::'):]\n    \n    if not bucket_name:\n        raise ValueError(\"Bucket name cannot be empty\")\n    \n    return bucket_name\n\n\n# Test cases\nif __name__ == \"__main__\":\n